# Web Crawler: Thu thập dữ liệu phòng trọ tại TP.HCM

Notebook này chuyển từ file `.py` sang dạng `.ipynb`, giữ nguyên logic cào dữ liệu từ [phongtro123.com](https://phongtro123.com) nhưng được tổ chức lại theo từng phần rõ ràng.

Mỗi cell code đều có phần giải thích bằng Markdown, theo style Data Scientist để dễ đọc, dễ maintain và tái sử dụng.

## 1. Import thư viện & cấu hình cơ bản

Ở bước đầu tiên, ta import các thư viện cần thiết:

- `re`, `json`, `csv`, `time`, `random`, `threading` cho xử lý chuỗi, tệp và đa luồng.
- `datetime` để chuẩn hóa thời gian đăng tin.
- `concurrent.futures.ThreadPoolExecutor` để cào chi tiết bằng multi-thread.
- `requests` + `BeautifulSoup` để gửi HTTP request và parse HTML.
- `urljoin` để ghép URL tương đối thành tuyệt đối.

Đồng thời khai báo một số hằng số cấu hình cho domain, số luồng, và user-agent.

In [ ]:
import re
import csv
import time
import random
import json
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


# =========================================================
# CONFIG
# =========================================================
BASE_DOMAIN = "https://phongtro123.com"
BASE_LIST_URL = "https://phongtro123.com/tinh-thanh/ho-chi-minh"
SOURCE_NAME = "phongtro123"

# Số luồng cào detail
MAX_WORKERS = 15  # tăng/giảm tuỳ máy + mức bị chặn
# Delay nhỏ để tránh bị block quá nhanh (0 = nhanh nhất)
DETAIL_DELAY_RANGE = (0.05, 0.2)

# Stop khi list trống liên tiếp
EMPTY_STOP = 3

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_6) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36",
]

## 2. Định nghĩa các Regex dùng để trích xuất thông tin

Phần này khai báo trước các biểu thức chính quy (regex) để:

- Nhận diện link bài đăng (`RE_POST_LINK`).
- Bắt số điện thoại Việt Nam (`RE_PHONE`).
- Tìm thời gian & ngày đăng tin.
- Nhận diện label "Ngày đăng".
- Tìm giá, diện tích với nhiều cách ghi khác nhau (triệu, nghìn, m², ...).

In [ ]:
# =========================================================
# REGEX
# =========================================================
RE_POST_LINK = re.compile(r"-pr\d+\.html?$", re.I)
RE_PHONE = re.compile(r"\b0\d{8,10}\b")
RE_TIME_DATE = re.compile(r"(\d{1,2}:\d{2})\s*(\d{1,2}/\d{1,2}/\d{4})")
RE_NGAY_DANG = re.compile(r"ngày\s*đăng\s*:?\s*([^\n]+)", re.I)

RE_PRICE_LABEL = re.compile(
    r"Giá[^0-9]{0,20}([0-9]{1,3}(?:[.,][0-9]{1,3})*(?:[.,][0-9]{1,2})?)\s*(triệu|tr|nghìn|ngàn|k|đ|vnđ|vnd)",
    re.I
)
RE_PRICE_ANY = re.compile(
    r"([0-9]{1,3}(?:[.,][0-9]{1,3})*(?:[.,][0-9]{1,2})?)\s*(triệu|tr|nghìn|ngàn|k|đ|vnđ|vnd)"
    r"(?:\s*\/\s*tháng|\s*tháng|\s*\/\s*người|\s*người)?",
    re.I
)

RE_AREA_LABEL = re.compile(
    r"Diện tích[^0-9]{0,20}([0-9]{1,3}(?:[.,][0-9]{1,2})?)\s*(m2|m²|met vuong|m vuong)",
    re.I
)
RE_AREA_ANY = re.compile(
    r"([0-9]{1,3}(?:[.,][0-9]{1,2})?)\s*(m2|m²|met vuong|m vuong)",
    re.I
)

## 3. Tạo `requests.Session` riêng cho từng thread

`requests.Session` không hoàn toàn thread-safe, vì vậy ta dùng `threading.local()` để
mỗi thread có một session riêng. Điều này giúp:

- Tái sử dụng connection (keep-alive) tốt hơn.
- Giảm overhead khi tạo session mới liên tục.
- Hạn chế xung đột giữa các luồng.

In [ ]:
# =========================================================
# THREAD-LOCAL SESSION (requests.Session không thread-safe)
# =========================================================
_thread_local = threading.local()

def get_session():
    """Khởi tạo hoặc lấy lại session gắn với từng thread."""
    if getattr(_thread_local, "session", None) is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(
            pool_connections=50,
            pool_maxsize=50,
            max_retries=0
        )
        s.mount("http://", adapter)
        s.mount("https://", adapter)
        _thread_local.session = s
    return _thread_local.session

## 4. Hàm gửi request HTTP

Hai helper chính:

- `make_headers`: Sinh HTTP header với User-Agent ngẫu nhiên, thêm Referer.
- `fetch`: Gửi request GET với retry nhẹ, kiểm tra một số pattern liên quan đến Cloudflare / JS.

Hàm `fetch` trả về HTML (string) nếu thành công, hoặc `None` nếu thất bại sau số lần retry.

In [ ]:
# =========================================================
# HTTP
# =========================================================
def make_headers(referer=None):
    """Sinh HTTP headers với User-Agent ngẫu nhiên."""
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Connection": "keep-alive",
        "Referer": referer or BASE_DOMAIN,
    }


def fetch(url, referer=None, retries=3, timeout=20):
    """Gửi request GET nhanh với session pool và retry nhẹ.

    - Dùng session theo thread để tận dụng keep-alive.
    - Retry một vài lần khi gặp lỗi mạng/HTTP.
    - Trả về nội dung HTML (str) hoặc None.
    """
    session = get_session()
    last_err = None
    for i in range(retries):
        try:
            r = session.get(url, headers=make_headers(referer), timeout=timeout)
            if r.status_code == 200 and r.text.strip():
                low = r.text.lower()
                if "just a moment" in low or "enable javascript" in low:
                    last_err = "cloudflare page"
                else:
                    return r.text
            else:
                last_err = f"status={r.status_code}"
        except Exception as e:
            last_err = str(e)

        # ngủ ngắn giữa các lần retry
        time.sleep(0.6 + i * 0.6)

    print(f"[WARN] fetch failed {last_err} at {url}")
    return None

## 5. Các hàm helpers xử lý chuỗi & JSON

Một số helper nhỏ hỗ trợ việc chuẩn hóa dữ liệu:

- `normalize_space`: gom nhiều khoảng trắng thành 1.
- `fix_unit_spacing`: chuẩn hóa cách ghi đơn vị (m2, triệu/tháng, ...).
- `pick_first_text`: lấy text đầu tiên tìm được theo danh sách CSS selector.
- `extract_by_label_text`: trích xuất giá trị ngay sau một label (VD: `Địa chỉ:`).
- `safe_json_load`, `find_in_json`: parse JSON trong các thẻ script.
- `iso_to_hhmm_ddmmyyyy`: chuyển ISO datetime về định dạng `HH:MM dd/mm/yyyy`.

In [ ]:
# =========================================================
# HELPERS
# =========================================================
def normalize_space(s: str):
    return re.sub(r"\s+", " ", (s or "")).strip()


def fix_unit_spacing(s: str):
    s = normalize_space(s)
    s = re.sub(r"\bm\s*2\b", "m2", s, flags=re.I)
    s = re.sub(r"\btriệu\s*\/\s*tháng\b", "triệu/tháng", s, flags=re.I)
    s = re.sub(r"\btr\s*\/\s*tháng\b", "tr/tháng", s, flags=re.I)
    return s


def pick_first_text(soup, selectors):
    """Lấy text đầu tiên tìm được trong danh sách CSS selector."""
    for sel in selectors:
        tag = soup.select_one(sel)
        if tag:
            txt = tag.get_text(" ", strip=True)
            if txt:
                return txt
    return ""


def extract_by_label_text(visible_text, labels):
    """Trích xuất phần text sau một label nhất định (VD: 'Địa chỉ:', 'Khu vực:')."""
    for lb in labels:
        m1 = re.search(rf"{lb}\s*:\s*([^\n]+)", visible_text, flags=re.I)
        if m1:
            val = m1.group(1).strip()
            if val:
                return val
        m2 = re.search(rf"{lb}\s*:\s*\n\s*([^\n]+)", visible_text, flags=re.I)
        if m2:
            val = m2.group(1).strip()
            if val:
                return val
    return ""


def safe_json_load(s):
    try:
        return json.loads(s)
    except Exception:
        return None


def find_in_json(obj, keys):
    """Tìm giá trị theo key trong cấu trúc JSON lồng nhau (dict/list)."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys and isinstance(v, (str, int, float)):
                return str(v)
            found = find_in_json(v, keys)
            if found:
                return found
    elif isinstance(obj, list):
        for it in obj:
            found = find_in_json(it, keys)
            if found:
                return found
    return ""


def iso_to_hhmm_ddmmyyyy(s):
    try:
        s = s.strip().replace("Z", "+00:00")
        dt = datetime.fromisoformat(s)
        return dt.strftime("%H:%M %d/%m/%Y")
    except Exception:
        return ""

## 6. Lọc bài đăng thuộc TP.HCM

Để đảm bảo chỉ cào tin thuộc TP.HCM, ta viết một hàm nhỏ nhận `address` và một phần text phụ
rồi kiểm tra các từ khóa quen thuộc như:
`Hồ Chí Minh`, `Ho Chi Minh`, `TP.HCM`, `TPHCM`.

In [ ]:
# =========================================================
# HCM FILTER
# =========================================================
def is_hcm_address(address: str, extra_text: str = "") -> bool:
    low = normalize_space((address or "") + " " + (extra_text or "")).lower()
    return bool(re.search(r"\b(hồ\s*chí\s*minh|ho\s*chi\s*minh|tp\.?\s*hcm|tphcm)\b", low))

## 7. Hàm lấy danh sách link bài đăng từ trang list

Hàm `get_post_links_from_list(page_number)` sẽ:

1. Ghép URL list theo `page_number`.
2. Gửi request qua `fetch`.
3. Dùng CSS selector để tìm các thẻ chứa bài đăng và trích URL.
4. Nếu không tìm được theo DOM, fallback sang regex để bắt link dạng `*-prxxxx.html`.

Kết quả trả về là `(list_links, list_url)`.

In [ ]:
# =========================================================
# LIST PAGE -> LINKS
# =========================================================
def get_post_links_from_list(page_number):
    list_url = f"{BASE_LIST_URL}?page={page_number}"
    html = fetch(list_url, referer=BASE_LIST_URL)
    if not html:
        return [], list_url

    soup = BeautifulSoup(html, "html.parser")
    links = set()

    post_items = soup.select(
        "li.post-item, article.post-item, div.post-item, "
        "li.post-listing-item, article.post-listing-item, div.post-listing-item"
    )

    for item in post_items:
        a = item.select_one("h3 a[href], h2 a[href], a[href]")
        if not a:
            continue
        href = a.get("href", "")
        if href and RE_POST_LINK.search(href):
            full = urljoin(BASE_DOMAIN, href.split("?")[0])
            low = full.lower()
            if any(x in low for x in ["/tags/", "/blog/", "/tin-tuc/"]):
                continue
            links.add(full)

    # fallback regex nếu không tìm được theo DOM
    if not links:
        for m in re.finditer(r'href="([^"]*-pr\d+\.html?)"', html):
            href = m.group(1)
            full = urljoin(BASE_DOMAIN, href.split("?")[0])
            low = full.lower()
            if any(x in low for x in ["/tags/", "/blog/", "/tin-tuc/"]):
                continue
            links.add(full)

    return list(links), list_url

## 8. Trích xuất mô tả, thời gian đăng, điện thoại, giá, diện tích

Các hàm trong phần này chịu trách nhiệm trích xuất thông tin chi tiết từ HTML:

- `extract_full_description`: gom phần mô tả đầy đủ của bài đăng.
- `extract_posted_time`: cố gắng lấy đúng **ngày đăng** (ưu tiên) thay vì ngày cập nhật.
- `extract_phone`: tìm số điện thoại trong thẻ `tel:` hoặc trong text.
- Các hàm xử lý đơn vị:
  - `ensure_month_unit`, `ensure_m2_unit`.
  - `extract_price`, `extract_area`: tìm giá và diện tích bằng DOM, JSON-LD, hoặc regex.

In [ ]:
# =========================================================
# DESCRIPTION
# =========================================================
def extract_full_description(soup):
    cont = soup.select_one(
        "div.post-main-content, section.post-main-content, "
        "div#post-content, div.post-content, "
        "div.post-description, section.post-description, "
        "div.description"
    )
    if cont:
        txt = cont.get_text("\n", strip=True)
        return normalize_space(txt.replace("\r", "\n"))

    heading = soup.find(
        lambda t: t.name in ["h2", "h3", "h4"]
        and "thông tin mô tả" in t.get_text(strip=True).lower()
    )
    if heading:
        texts = []
        node = heading
        while True:
            node = node.find_next_sibling()
            if not node:
                break
            cls = " ".join(node.get("class", [])).lower()
            if node.name in ["h2", "h3", "h4"] and "thông tin mô tả" not in node.get_text(strip=True).lower():
                break
            if any(k in cls for k in ["post-attributes", "box-user-info", "user-info", "section-contact", "post-summary"]):
                break
            t = node.get_text("\n", strip=True)
            if t:
                texts.append(t)
        if texts:
            return normalize_space("\n".join(texts))

    return ""


# =========================================================
# POSTED TIME (ƯU TIÊN NGÀY ĐĂNG)
# =========================================================
def extract_posted_time(soup, visible_text):
    # Tìm theo label 'Ngày đăng'
    m = RE_NGAY_DANG.search(visible_text)
    if m:
        raw = m.group(1).strip()
        mm = RE_TIME_DATE.search(raw)
        if mm:
            return f"{mm.group(1)} {mm.group(2)}"
        return raw

    # Tìm trong các dòng thuộc post-attributes / summary
    for row in soup.select(
        ".post-attributes li, .post-attributes .item, "
        ".post-summary .summary-item, .summary-item"
    ):
        t = row.get_text(" ", strip=True)
        if t and re.search(r"ngày\s*đăng", t, re.I):
            mm = RE_TIME_DATE.search(t)
            if mm:
                return f"{mm.group(1)} {mm.group(2)}"
            t2 = re.sub(r"ngày\s*đăng\s*:?\s*", "", t, flags=re.I).strip()
            if t2:
                return t2

    # Thẻ <time>
    time_tag = soup.find("time")
    if time_tag:
        raw = time_tag.get_text(" ", strip=True)
        if raw and not re.search(r"cập\s*nhật", raw, re.I):
            mm = RE_TIME_DATE.search(raw)
            if mm:
                return f"{mm.group(1)} {mm.group(2)}"
            dt_attr = time_tag.get("datetime")
            if dt_attr:
                formatted = iso_to_hhmm_ddmmyyyy(dt_attr)
                if formatted:
                    return formatted
            return raw

    # Meta published_time
    meta = soup.select_one("meta[property='article:published_time']")
    if meta and meta.get("content"):
        formatted = iso_to_hhmm_ddmmyyyy(meta["content"])
        if formatted:
            return formatted

    # Fallback: tìm pattern thời gian/ngày trong toàn bộ text
    matches = list(RE_TIME_DATE.finditer(visible_text))
    if matches:
        for mt in matches:
            start = max(0, mt.start() - 30)
            window = visible_text[start:mt.start()].lower()
            if "cập nhật" not in window:
                return f"{mt.group(1)} {mt.group(2)}"
        first = matches[0]
        return f"{first.group(1)} {first.group(2)}"

    return ""


# =========================================================
# PHONE
# =========================================================
def extract_phone(soup, visible_text):
    tel = soup.select_one('a[href^="tel:"]')
    if tel:
        num = tel.get("href", "").replace("tel:", "").strip()
        if num:
            return re.sub(r"\D", "", num)
    m = RE_PHONE.search(visible_text.replace(" ", ""))
    return m.group(0) if m else ""


# =========================================================
# PRICE / AREA
# =========================================================
def ensure_month_unit(price_text: str):
    if not price_text:
        return ""
    low = price_text.lower()
    if any(x in low for x in ["tháng", "/thang", "người", "/nguoi", "m2", "m²"]):
        return price_text
    return price_text + "/tháng"


def ensure_m2_unit(area_text: str):
    if not area_text:
        return ""
    low = area_text.lower()
    if "m2" in low or "m²" in low or "vuong" in low:
        return area_text
    if re.fullmatch(r"[0-9.,]+", area_text.strip()):
        return area_text.strip() + " m2"
    return area_text


def extract_price(soup, visible_text, title="", desc=""):
    price = pick_first_text(soup, [
        "span.item-price",
        ".post-summary .summary-item .price",
        ".post-attributes .price",
        "span.price",
        "div.price"
    ])
    if price:
        return ensure_month_unit(fix_unit_spacing(price))

    for sc in soup.select('script[type="application/ld+json"]'):
        data = safe_json_load(sc.string or sc.get_text())
        if data:
            v = find_in_json(data, keys={"price", "lowPrice", "highPrice"})
            if v:
                return ensure_month_unit(fix_unit_spacing(v))

    m = RE_PRICE_LABEL.search(visible_text) or RE_PRICE_ANY.search(visible_text)
    if m:
        return ensure_month_unit(fix_unit_spacing(f"{m.group(1)} {m.group(2)}"))

    mix = f"{title}\n{desc}"
    m = RE_PRICE_LABEL.search(mix) or RE_PRICE_ANY.search(mix)
    if m:
        return ensure_month_unit(fix_unit_spacing(f"{m.group(1)} {m.group(2)}"))

    return ""


def extract_area(soup, visible_text, title="", desc=""):
    area = pick_first_text(soup, [
        "span.item-area",
        ".post-summary .summary-item .acreage",
        ".post-attributes .acreage",
        "span.acreage",
        "div.acreage"
    ])
    if area:
        return ensure_m2_unit(fix_unit_spacing(area))

    m = RE_AREA_LABEL.search(visible_text) or RE_AREA_ANY.search(visible_text)
    if m:
        return ensure_m2_unit(fix_unit_spacing(f"{m.group(1)} m2"))

    mix = f"{title}\n{desc}"
    m = RE_AREA_LABEL.search(mix) or RE_AREA_ANY.search(mix)
    if m:
        return ensure_m2_unit(fix_unit_spacing(f"{m.group(1)} m2"))

    return ""

## 9. Trích xuất tên người đăng (owner_name)

Để lấy tên chủ tin đăng, ta:

- Tìm trong các khu vực liên hệ / user-info.
- Loại bỏ những đoạn chứa số điện thoại, từ khóa như `Zalo`, `liên hệ`, ...
- Nếu không có trong DOM rõ ràng, fallback sang cách duyệt text xung quanh label "Thông tin liên hệ".

In [ ]:
# =========================================================
# OWNER NAME
# =========================================================
RE_OWNER_JUNK = re.compile(
    r"(zalo|mobi|sđt|điện thoại|liên hệ|nhắn|tin đăng|tham gia|đang hoạt động)",
    re.I
)


def clean_owner_name(raw: str):
    if not raw:
        return ""
    raw = re.sub(r"\d[\d\.\s]{6,}\d", " ", raw)
    raw = re.sub(r"\b0\d{8,10}\b", " ", raw)
    raw = RE_OWNER_JUNK.sub(" ", raw)
    raw = normalize_space(raw)
    raw = re.sub(r"\d+", " ", raw)
    return normalize_space(raw)


def extract_owner_name(soup, page_text):
    candidates = []
    selectors = [
        ".post-contact .contact-name",
        ".post-contact .name",
        ".box-user-info .name",
        ".box-user-info .user-name",
        ".contact-info .name",
        ".user-info .name",
        ".user-info .user-name",
        ".post-author .author-name",
        ".author-name",
        "div.user-info strong",
        "div.user-info h3",
        "div.user-info h4",
    ]
    for sel in selectors:
        for tag in soup.select(sel):
            txt = clean_owner_name(tag.get_text(" ", strip=True))
            if txt and len(txt) >= 2:
                candidates.append(txt)

    if candidates:
        # lấy chuỗi ngắn nhất sau khi loại rác
        return sorted(set(candidates), key=len)[0]

    # Tìm section theo heading 'Thông tin liên hệ'
    heading = soup.find(lambda t: t.name in ["h2","h3","h4"] and "Thông tin liên hệ" in t.get_text())
    if heading:
        section = heading.find_parent(["section","div","aside"]) or heading.parent
        if section:
            for s in section.stripped_strings:
                s = normalize_space(s)
                if not s or RE_PHONE.search(s) or RE_OWNER_JUNK.search(s):
                    continue
                s2 = clean_owner_name(s)
                if s2 and len(s2) >= 2:
                    return s2

    # Fallback: tìm quanh dòng 'Thông tin liên hệ' trong toàn bộ text
    lines = [normalize_space(x) for x in page_text.split("\n") if normalize_space(x)]
    for i, line in enumerate(lines):
        if "Thông tin liên hệ" in line:
            for j in range(i+1, min(i+8, len(lines))):
                cand = clean_owner_name(lines[j])
                if cand and len(cand) >= 2:
                    return cand

    return ""

## 10. Hàm parse chi tiết một bài đăng

Hàm `parse_detail(post_url, referer)` sẽ:

1. Gửi request lấy HTML bài đăng.
2. Parse bằng BeautifulSoup.
3. Lấy title, địa chỉ, mô tả, giá, diện tích, thời gian đăng, chủ tin, số điện thoại.
4. Lọc chỉ giữ bài ở TP.HCM.
5. Trả về dict dữ liệu chuẩn hóa (hoặc `None` nếu bài không hợp lệ).

In [ ]:
# =========================================================
# DETAIL PARSER (CHẠY TRONG LUỒNG)
# =========================================================
def parse_detail(post_url, referer):
    html = fetch(post_url, referer=referer)
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")
    visible_text = soup.get_text("\n", strip=True)

    title = pick_first_text(soup, [
        "h1.page-h1", "h1.post-title-lg", "h1.post-title", "h1"
    ])

    address = pick_first_text(soup, [
        "address.post-address",
        "div.post-address",
        "span.post-address",
        ".post-summary address",
        ".summary-item.address",
        "address"
    ]) or extract_by_label_text(visible_text, ["Địa chỉ", "Khu vực"])

    # Lọc các tin không thuộc TP.HCM
    if not is_hcm_address(address, visible_text):
        return None

    description = extract_full_description(soup)
    price = extract_price(soup, visible_text, title=title, desc=description)
    area  = extract_area(soup, visible_text, title=title, desc=description)

    posted_time = extract_posted_time(soup, visible_text)
    owner_name = extract_owner_name(soup, visible_text)
    phone = extract_phone(soup, visible_text)

    # Các rule loại bỏ tin không đủ thông tin
    if not title:
        return None
    if (not price) and (not area):
        return None

    # Delay rất nhỏ để tránh bị chặn quá nhanh
    if DETAIL_DELAY_RANGE and DETAIL_DELAY_RANGE != (0, 0):
        time.sleep(random.uniform(*DETAIL_DELAY_RANGE))

    return {
        "url": post_url,
        "title": title,
        "price": price,
        "area": area,
        "address": address,
        "description": description,
        "posted_time": posted_time,
        "owner_name": owner_name,
        "phone": phone,
        "source": SOURCE_NAME
    }

## 11. Hàm chính `crawl_phongtro123`

Hàm `crawl_phongtro123` là pipeline chính để cào dữ liệu:

1. Duyệt tuần tự từng trang list (`start_page` → `end_page` hoặc tới khi hết dữ liệu).
2. Lấy danh sách link bài đăng, loại trùng.
3. Dùng `ThreadPoolExecutor` để parse detail song song.
4. Ghi kết quả từng bản ghi vào file CSV.
5. Có thể dừng khi đạt `target_rows`.

Ở đây mình **đã bỏ hết icon/emojis** trong các câu `print` để log gọn gàng, phù hợp chạy production/log file.

In [ ]:
# =========================================================
# MAIN CRAWLER (LIST SEQ + DETAIL PARALLEL)
# =========================================================
def crawl_phongtro123(start_page=1, end_page=None, output_csv="phongtro123.csv",
                      target_rows=None, empty_stop=EMPTY_STOP):

    fieldnames = [
        "url","title","price","area",
        "address","description",
        "posted_time","owner_name","phone","source"
    ]

    seen = set()
    written = 0
    empty_streak = 0
    page = start_page

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

        # Executor sống xuyên suốt để reuse thread + session pool
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            while True:
                if end_page is not None and page > end_page:
                    print("Reached end_page. Stop.")
                    break

                links, list_url = get_post_links_from_list(page)
                print(f"[LIST] Page {page}: {list_url}")
                print(f"  -> found {len(links)} post links")

                if not links:
                    empty_streak += 1
                    if end_page is None and empty_streak >= empty_stop:
                        print("No more pages. Stop.")
                        break
                    page += 1
                    continue
                empty_streak = 0

                # Lọc trùng trước khi submit
                new_links = []
                for link in links:
                    if link not in seen:
                        seen.add(link)
                        new_links.append(link)

                if not new_links:
                    page += 1
                    continue

                # Submit đa luồng cào detail
                futures = [ex.submit(parse_detail, link, list_url) for link in new_links]

                for fut in as_completed(futures):
                    item = None
                    try:
                        item = fut.result()
                    except Exception as e:
                        print(f"    [ERR] detail exception: {e}")

                    if item:
                        w.writerow(item)
                        written += 1
                        print(
                            f"    OK: {item['posted_time']} | {item['address']} | "
                            f"{item['price']} | {item['area']} | {item['owner_name']}"
                        )

                    if target_rows is not None and written >= target_rows:
                        print(f"Reached target_rows={target_rows}. Stop.")
                        return

                page += 1

    print(f"Done. rows written={written}, unique urls={len(seen)} -> {output_csv}")

## 12. Chạy thử crawler (tuỳ chọn)

Cell dưới đây minh họa cách gọi hàm `crawl_phongtro123`.

**Lưu ý:**

- Khi chạy thực tế, nên bắt đầu với số trang nhỏ để kiểm tra logic.
- Sau khi ổn, có thể mở rộng `end_page` hoặc đặt `target_rows` lớn hơn.
- Không nên spam server quá nhanh, hãy tôn trọng robots.txt và điều chỉnh `MAX_WORKERS`, `DETAIL_DELAY_RANGE` phù hợp.

In [ ]:
# Ví dụ: cào vài trang đầu để test
# Gợi ý: nên thử với end_page nhỏ trước (VD: 2 hoặc 3)

crawl_phongtro123(start_page=1, end_page=2, output_csv="phongtro123_sample.csv")
pass